# 3.8 Ablation (CIFAR-10, PreAct ResNet-18) trên Kaggle

Chạy các dòng của Bảng 5 bằng `experiments/exp38_ablation.py`, rồi gom kết quả bằng
`experiments/exp38_report.py`. Mỗi lần chạy lưu `last.pt` sau từng epoch nên bị ngắt thì chạy lại là tiếp tục.

**Trước khi chạy:** Settings → Accelerator **GPU T4 x2**, Internet **On**, Add Input → dataset chứa thư mục `Source`.
Nếu chạy tiếp phiên cũ: Add Input thêm **output của version mới nhất** (chỉ version chạy CÙNG giao thức;
**không** thêm output của lần chạy 100 epoch/AMP cũ, vì tên thư mục trùng nên notebook sẽ tưởng là đã xong).

**Cách dùng:** chọn `RUNS` của tài khoản mình ở ô tham số, rồi **Save Version → Save & Run All (Commit)**.
Notebook chạy thử nhanh, đo thời gian, rồi chạy thật; lỗi ở bước chạy thử thì dừng ngay.
Gom kết quả + ERM/mixup của 3.2: `python experiments/exp38_report.py --results <thư mục results> --import-exp32 <kết quả 3.2>`.


In [ ]:
# ==== Tham số: chỉnh ở đây ====
# Giao thức GIỐNG 3.2 của Nhi (200 epoch, FP32) để dùng lại ERM + mixup wd 1e-4 của Nhi
EPOCHS, MILESTONES = 200, "100 150"
AMP      = False

# Đợt 1 (seed 0) = 10 ô Bảng 5; 2 ô (erm, 1e-4) và (mixup, 1e-4) lấy từ 3.2 của Nhi -> 8 ô tự chạy,
# chia 2 tài khoản Kaggle, mỗi tài khoản ~10 giờ: (biến thể, weight decay, seed)
RUNS = [("input_ac_rp", 1e-4, 0), ("ls_0.1", 1e-4, 0), ("input_ls_0.2", 1e-4, 0),
        ("erm", 5e-4, 0)]                                                      # tài khoản 1
# RUNS = [("mixup", 5e-4, 0), ("input_ac_rp", 5e-4, 0), ("ls_0.1", 5e-4, 0),
#         ("input_ls_0.2", 5e-4, 0)]                                         # tài khoản 2

# Khi RUNS = None thì chạy đủ VARIANTS x WDS x SEEDS
VARIANTS = ["erm", "mixup", "input_ac_rp", "ls_0.1", "input_ls_0.2"]
WDS      = [1e-4, 5e-4]
SEEDS    = [0]

SMOKE_VARIANTS = ["erm", "mixup", "input_ac_rp", "ls_0.1", "input_ls_0.2"]
SMOKE        = True    # 1. chạy thử nhanh (~3 phút) trước khi chạy thật
MEASURE_TIME = True    # 2. đo giây/epoch (FP32 và AMP), ~5 phút
RUN_FULL     = True    # 3. chạy thật -> dùng Save Version -> Save & Run All (Commit)
WORKERS      = 4


In [ ]:
import os, sys, glob, shutil, subprocess, time, json
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ")
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/core/mixup.py', recursive=True)]
assert cands, "Không thấy Source. Add Input -> chọn dataset chứa thư mục Source"
SRC = '/kaggle/working/Source'
if not os.path.exists(SRC):
    shutil.copytree(cands[0], SRC)
os.chdir(SRC)
DATA, OUT = '/kaggle/working/data', '/kaggle/working/results'
# Chạy tiếp phiên cũ: chép results cũ (nếu có trong input) vào working
for old in glob.glob('/kaggle/input/**/results/exp38_ablation', recursive=True):
    dst = f"{OUT}/exp38_ablation"
    if not os.path.exists(dst):
        shutil.copytree(old, dst)
        print("Đã chép kết quả cũ từ", old)


In [ ]:
!python -m pytest tests -q || python tests/test_core.py

In [ ]:
def run(args):
    cmd = (f"python experiments/exp38_ablation.py --data-dir {DATA} --out-dir {OUT} --workers {WORKERS} "
           f"--epochs {EPOCHS} --milestones {MILESTONES} " + ("--amp " if AMP else "") + args)
    print("$", cmd, flush=True)
    t = time.time()
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    print(f"-> xong sau {time.time()-t:.0f}s, exit={rc}", flush=True)
    assert rc == 0, "Lệnh lỗi, xem log phía trên"


## 1. Chạy thử từng biến thể (khoảng 10 phút)

In [ ]:
if SMOKE:
    for v in SMOKE_VARIANTS:
        run(f"--variant {v} --seed 0 --wd 1e-4 --epochs 1 --limit-batches 20 --no-resume --run-name smoke38/{v}")
    print("\nCHẠY THỬ OK: mọi biến thể đều chạy được")


## 2. Đo thời gian một epoch đầy đủ

Chạy mixup 2 epoch đầy đủ (FP32 rồi AMP). Bỏ epoch đầu vì còn khởi động. Kết quả dùng để chọn `AMP`
và chia các đợt sao cho mỗi đợt dưới 12 giờ (giới hạn một lần Commit của Kaggle).

In [ ]:
if MEASURE_TIME:
    import pandas as pd
    rows = []
    for mode, flag in [("fp32", ""), ("amp", "--amp")]:
        name = f"smoke38/timing_{mode}"
        run(f"--variant mixup --seed 0 --wd 1e-4 --epochs 2 --no-resume --run-name {name} {flag}")
        t = pd.read_csv(f"{OUT}/{name}/history.csv").epoch_time_s.iloc[-1]
        n_runs = len(RUNS) if RUNS else len(VARIANTS) * len(WDS) * len(SEEDS)
        rows.append({"chế độ": mode, "giây/epoch": round(t, 1),
                     "giờ/1 lần chạy": round(t * EPOCHS / 3600, 2),
                     f"giờ cho {n_runs} lần chạy": round(t * EPOCHS * n_runs / 3600, 1)})
    est = pd.DataFrame(rows)
    est["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    display(est)
    est.to_csv(f"{OUT}/smoke38/timing_estimate.csv", index=False)


## 3. Chạy thật

Thứ tự: theo seed, rồi weight decay, rồi biến thể, để nếu hết giờ thì đã có đủ một seed cho mọi ô.
Lần chạy đã xong (có `summary.json`) sẽ được bỏ qua; lần chạy dở sẽ tự resume.
Chạy xong lần nào thì xóa file model `.pt` của lần đó (bảng 3.8 chỉ cần log), để output nhẹ và chép sang phiên sau nhanh.

In [ ]:
if RUN_FULL:
    plan = RUNS or [(v, wd, seed) for seed in SEEDS for wd in WDS for v in VARIANTS]
    for v, wd, seed in plan:
        d = f"{OUT}/exp38_ablation/{v}_wd{wd:g}_seed{seed}"
        if os.path.exists(f"{d}/summary.json"):
            print("bỏ qua (đã xong):", d)
            continue
        run(f"--variant {v} --seed {seed} --wd {wd:g}")
        for f in glob.glob(f"{d}/*.pt"):
            os.remove(f)
else:
    print("RUN_FULL = False: không chạy thật")


## 4. Bảng so sánh với Bảng 5 + biểu đồ

In [ ]:
if os.path.isdir(f"{OUT}/exp38_ablation"):
    !python experiments/exp38_report.py --results {OUT}
    from IPython.display import Image, display
    for p in sorted(glob.glob(f"{OUT}/exp38_ablation/report_wd*.png")):
        display(Image(p))


In [ ]:
# Đóng gói log (không kèm .pt) để tải từ tab Output. Giữ nguyên thư mục results nếu cần chạy tiếp.
!cd /kaggle/working && zip -qr exp38_logs.zip results -x '*.pt' && ls -lh exp38_logs.zip